# Suite MOD — Modules: schemas as data

A module is a named set of schemas held by an object of the meta-schema `Schemas.Module.Schema`, registered as
`Schemas.Module`. `Modules.module(schemas)` builds one and `Modules.schemas(module)` reads one back, so schemas are
written, read, validated and compared like any other objects. Within a module each schema is written inline, by kind,
and refers by name to the schemas in the module and those registered.

In [ ]:
import { Comparison as C, JSON, Modules, Plain, Proxies, Schemas as S, Validators, YAML } from "@mbse/schemas/Framework";
import { LookupError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, map, raises, same, text } from "./support.js";

const store = new Proxies.OfStore();

const B = store;
const Int32 = S.OfNative.resolve((n) => n.name("Int32").type(BigInt).bits(32n));
const Size = S.OfNative.resolve((n) => n.name("Size").token("ccpp", "size_t").bytes(8n));
const Phone = new S.OfObject.Builder().name("Phone").properties(text("number")).create();
const Reach = new S.OfUnion.Builder().branches((b) => b.name("phone").of(Phone),
  (b) => b.name("other").of(new S.OfObject.Builder().properties(text("note")).create())).create();
const Stamped = new S.OfIntersection.Builder().parts((p) => p.name("a").of(Phone), (p) => p.name("b").of(Phone)).create();
const Contact = new S.OfObject.Builder().name("Contact").ref().properties(
  text("name"), (p) => p.name("home").of(Phone), (p) => p.name("reach").of(Reach),
  (p) => p.name("stamped").of(Stamped), (p) => p.name("ids").of((t) => t.as_indexed((i) => i.of(Int32)))).create();
const Directory = new S.OfObject.Builder().name("Directory").ref().singleton("Directory").create();
const Listing = new S.OfRelation.Builder().name("Listing").links("directory", "contact").properties(text("key")).unique(
  "directory", "key").unique("contact").create();
const Labels = new S.OfRelation.Builder().name("Labels").links("contact", "phone").create(); // registered, not in the module, below
new S.OfObject.Builder(Contact).relations((a) => a.name("listings").of(Listing).me("contact"),
  (a) => a.name("labels").of(Labels).me("contact")).update();
new S.OfObject.Builder(Directory).relations((a) => a.name("listings").of(Listing).me("directory")).update();
store.register(Labels);
const SCHEMAS = [Int32, Size, Phone, Contact, Directory, Listing];

/** A module's entries, as plain data. */
function entries(module: unknown): any {
  const plain = Plain.ToPlain(store)(S.Module.Schema, module as never) as any;
  return plain.get("objects").get(plain.get("root")).get("schemas") ?? [];
}

/** A module decoded from plain entries, which `Modules.module` would not write. */
function module_of(...items: unknown[]): unknown {
  return Plain.FromPlain(B)(S.Module.Schema, map({ root: "s0", objects: { s0: { schemas: items } } }));
}

function named(name: string) {
  return { named: { name } };
}

const STR = { native: { format: "basic", token: "str" } };

## MOD-01 · The meta-schemas validate, and `Schemas.Module` is registered as a reference object schema

In [ ]:
{
  const meta = [S.Module.Schema, S.Module.Entry, S.Module.Definition, S.OfAny.Schema, S.OfAny.Named, S.OfNative.Schema,
    S.OfProperty.Schema, S.OfObject.Schema, S.OfAdjacency.Schema, S.OfRelation.Schema, S.OfRelation.Ref,
    S.OfUnion.Schema, S.OfIntersection.Schema, S.OfIndexed.Schema];
  assert(meta.every((schema) => schema.validate().length === 0));
  assert(Modules.MODULE === "Schemas.Module" && store.schema("Schemas.Module") === S.Module.Schema && S.Module.Schema.ref);
  assert(store.registered("Schemas.Module") === S.Module.Schema && store.registered("Labels") === Labels);
  raises(LookupError, () => store.registered("nope"), "no schema registered as 'nope'");
  const kinds = ["native", "object", "union", "intersection", "indexed", "apply"];
  assert(same(S.OfAny.Schema.branches.map((b) => b.name), [...kinds, "named"])); // a type: a schema of any kind, or a name
  assert(same(S.Module.Definition.branches.map((b) => b.name), [...kinds, "relation"]));
  assert(same(S.OfRelation.Ref.branches.map((b) => b.name), ["relation", "named"]));
  assert(same([...S.OfObject.Schema.properties.keys()], ["name", "parameters", "properties", "adjacencies", "singleton", "ref", "description"]));
  assert(same([...S.OfRelation.Schema.properties.keys()], ["name", "parameters", "links", "properties", "uniques", "description"]));
  assert(same([...S.OfNative.Schema.properties.keys()], ["name", "parameters", "format", "token", "bits", "bytes", "terms", "description"]));
}

## MOD-02 · A module writes each schema inline, by kind, and refers by name to the schemas in it and those registered

In [ ]:
{
  const module = Modules.module(store, SCHEMAS) as any;
  assert(module.schema_name() === "Schemas.Module" && Validators.Validate(B)(S.Module.Schema, module).length === 0);
  assert(Modules.module(B, SCHEMAS) !== module);
  const phone = { object: { properties: [{ name: "number", type: STR }] } };
  assert(equal(entries(module), map([
    { name: "Int32", schema: { native: { format: "basic", token: "int", bits: 32n } } },
    { name: "Size", schema: { native: { format: "ccpp", token: "size_t", bytes: 8n } } },
    { name: "Phone", schema: phone },
    { name: "Contact", schema: { object: {
      properties: [
        { name: "name", type: STR },
        { name: "home", type: named("Phone") },
        { name: "reach", type: { union: { branches: [
          { name: "phone", type: named("Phone") },
          { name: "other", type: { object: { properties: [{ name: "note", type: STR }] } } }] } } },
        { name: "stamped", type: { intersection: { parts: [
          { name: "a", type: named("Phone") }, { name: "b", type: named("Phone") }] } } },
        { name: "ids", type: { indexed: { item: named("Int32") } } }],
      adjacencies: [{ name: "listings", relation: named("Listing"), me: "contact" },
        { name: "labels", relation: named("Labels"), me: "contact" }], // registered
      ref: true } } },
    { name: "Directory", schema: { object: { adjacencies: [
      { name: "listings", relation: named("Listing"), me: "directory" }], singleton: "Directory", ref: true } } },
    { name: "Listing", schema: { relation: { links: ["directory", "contact"], properties: [{ name: "key", type: STR }],
      uniques: [["directory", "key"], ["contact"]] } } }])));
  // What has no name is written inline wherever it is used: a relation too. Empty contents are left out.
  const Pair = new S.OfRelation.Builder().links("a", "b").create();
  const Lone = new S.OfObject.Builder().name("Lone").relations((a) => a.name("pairs").of(Pair).me("a")).create();
  assert(equal(entries(Modules.module(store, [Lone, new S.OfUnion.Builder().name("Empty").create()])), map([
    { name: "Lone", schema: { object: { adjacencies: [
      { name: "pairs", relation: { relation: { links: ["a", "b"] } }, me: "a" }] } } },
    { name: "Empty", schema: { union: {} } }])));
  assert(equal(entries(Modules.module(store, [])), []) && Modules.schemas(store, B["Schemas.Module"]().create()).size === 0);
  // The module is an object like any other: written and read as JSON and YAML, and compared.
  const text_ = JSON.ToJSON(store)(S.Module.Schema, module);
  assert(JSON.ToJSON(store)(S.Module.Schema, JSON.FromJSON(B)(S.Module.Schema, text_) as any) === text_);
  assert(YAML.ToYAML(store)(S.Module.Schema, YAML.FromYAML(B)(S.Module.Schema, YAML.ToYAML(store)(S.Module.Schema, module)) as any) ===
    YAML.ToYAML(store)(S.Module.Schema, module));
  assert(new C.OfObject(S.Module.Schema, module).compare(new C.OfObject(S.Module.Schema, Modules.module(store, SCHEMAS) as any)) === 0);
  assert(new C.OfObject(S.Module.Schema, module).compare(new C.OfObject(S.Module.Schema, Modules.module(store, [Phone]) as any)) === null);
}

## MOD-03 · Reading a module back gives the same schemas, sharing what is named, recursive ones included

In [ ]:
{
  const module = Modules.module(store, SCHEMAS) as any;
  const back = Modules.schemas(store, JSON.FromJSON(B)(S.Module.Schema, JSON.ToJSON(store)(S.Module.Schema, module))) as Map<string, any>;
  assert(same([...back.keys()], SCHEMAS.map((schema) => schema.name)) && [...back.values()].every((schema) => !SCHEMAS.includes(schema)));
  assert([...back].every(([name, schema]) => schema.name === name)); // each read back under its name
  assert([...back.values()].every((schema) => schema.validate().length === 0));
  assert(JSON.ToJSON(store)(S.Module.Schema, Modules.module(store, back.values()) as any) === JSON.ToJSON(store)(S.Module.Schema, module));
  const contact = back.get("Contact");
  assert(contact.properties.get("home")?.type === back.get("Phone") && contact.properties.get("reach")?.type.branches[0].type === back.get("Phone"));
  assert(contact.properties.get("stamped")?.type.parts[1].type === back.get("Phone") && contact.properties.get("ids")?.type.item === back.get("Int32"));
  assert(contact.adjacencies.get("listings").relation === back.get("Listing") &&
    back.get("Listing") === back.get("Directory").adjacencies.get("listings").relation);
  assert(contact.adjacencies.get("labels").relation === Labels); // resolved in the store
  assert(contact.adjacencies.get("listings").me === "contact" && contact.ref && back.get("Directory").singleton === "Directory");
  assert(back.get("Int32").type === BigInt && back.get("Int32").bits === 32n && back.get("Size").type === null &&
    back.get("Size").bytes === 8n);
  assert(same(back.get("Listing").links, ["directory", "contact"]) &&
    equal(back.get("Listing").uniques.map((u: Set<string>) => [...u]), [["directory", "key"], ["contact"]]));
  // A schema that refers to itself is written once, by name, and read back as itself.
  const Node = new S.OfObject.Builder().name("Node").properties(text("label")).create();
  new S.OfObject.Builder(Node).properties((p) => p.name("children").of((t) => t.as_indexed((i) => i.of(Node)))).update();
  const tree = Modules.schemas(store, Modules.module(store, [Node])).get("Node") as S.OfObject.Data;
  assert((tree.properties.get("children")?.type as S.OfIndexed.Data).item === tree && tree.validate().length === 0);
  // Validating a schema that holds itself ends, and reports each problem once: as a property, an entry's or in a union.
  new S.OfObject.Builder(Node).properties((p) => p.name("w").of((t) => t.as_native((n) => n.type(BigInt).bits(0n)))).update();
  assert(same(Node.validate(), ["property 'w': a width in bits must be a positive int or a term, got 0"]));
  assert(new S.OfRelation.Builder().links("a", "b").properties((p) => p.name("tree").of(tree)).create().validate().length === 0);
  const Loop = new S.OfUnion.Builder().create();
  new S.OfUnion.Builder(Loop).branches((b) => b.name("more").of((t) => t.as_indexed((i) => i.of(Loop))),
    (b) => b.name("ints").of((t) => t.as_indexed((i) => i.of(Int32)))).update();
  assert(new S.OfObject.Builder().properties((p) => p.name("loop").of(Loop)).create().validate().length === 0);
  // Keys and extents are written and read back.
  const Grid = new S.OfObject.Builder().name("Grid").properties(
    (p) => p.name("attrs").of((t) => t.as_indexed((i) => i.key((t) => t.as_native(String)).of(Int32))),
    (p) => p.name("row").of((t) => t.as_indexed((i) => i.of(Int32).extent({ minimum: 1n, maximum: 3n }))),
    (p) => p.name("open").of((t) => t.as_indexed((i) => i.of(Int32).extent({ minimum: 2n })))).create();
  const grid = Modules.schemas(store, Modules.module(store, [Grid, Int32])).get("Grid") as S.OfObject.Data;
  const property = (name: string) => grid.properties.get(name)?.type as S.OfIndexed.Data;
  assert((property("attrs").key as S.OfNative.Data).type === String && property("attrs").extent === null);
  assert(property("row").extent?.equals(new S.OfIndexed.Extent(1n, 3n)) === true && property("row").key === null);
  assert(property("open").extent?.equals(new S.OfIndexed.Extent(2n, null)) === true);
  const written = Modules.schemas(store, module_of({ name: "L", schema: { indexed: { item: STR, extent: { maximum: 3n } } } })).get("L") as S.OfIndexed.Data;
  assert(written.extent?.equals(new S.OfIndexed.Extent(0n, 3n)) === true); // a minimum left out is 0
  // Inline schemas are read as new schemas of their kind.
  const Pair = new S.OfRelation.Builder().links("a", "b").create();
  const Lone = new S.OfObject.Builder().name("Lone").relations((a) => a.name("pairs").of(Pair).me("a")).create();
  const lone = Modules.schemas(store, Modules.module(store, [Lone])).get("Lone") as S.OfObject.Data;
  const relation = lone.adjacencies.get("pairs")?.relation as S.OfRelation.Data;
  assert(same(relation.links, ["a", "b"]) && relation !== Pair);
}

## MOD-04 · Refused: a schema referring to itself unnamed, an unknown name, a name of the wrong kind, a name defined twice

In [ ]:
{
  const Node = new S.OfObject.Builder().properties(text("label")).create();
  new S.OfObject.Builder(Node).properties((p) => p.name("next").of((t) => t.as_indexed((i) => i.of(Node)))).update();
  const Holder = new S.OfObject.Builder().name("Holder").properties((p) => p.name("root").of(Node)).create();
  raises(ValueError, () => Modules.module(store, [Holder]), "a schema that refers to itself must be named");
  raises(ValueError, () => Modules.module(store, [Phone, new S.OfUnion.Data()]),
    "a module holds named schemas; name each with its builder's .name()");
  raises(TypeError, () => Modules.module(store, [new S.OfObject.Data({ properties: new Map([["p", new S.OfProperty.Data({ name: "p", type: "text" as never })]]), name: "Bad" })]),
    "not a schema: 'text'");
  raises(TypeError, () => Modules.module(store, [new S.OfNative.Data("weird", { name: "Bad" })]), "unsupported native type 'weird'");
  const property_named = (name: string) => ({ object: { properties: [{ name: "p", type: named(name) }] } });
  raises(LookupError, () => Modules.schemas(store, module_of({ name: "A", schema: property_named("Nope") })),
    "no schema named 'Nope' in the module or the store");
  raises(TypeError, () => Modules.schemas(store, module_of({ name: "A", schema: property_named("Labels") })),
    "'Labels' is a relation, not a type");
  raises(TypeError, () => Modules.schemas(store, module_of({ name: "A", schema: { object: { adjacencies: [
    { name: "x", relation: named("A"), me: "a" }] } } })), "'A' is not a relation");
  raises(ValueError, () => Modules.schemas(store, module_of({ name: "A", schema: { union: {} } }, { name: "A", schema: { union: {} } })),
    "the module defines 'A' twice");
}

## MOD-05 · One type by name or inline: `reference` and `resolve`

In [ ]:
{
  const local = new Proxies.OfStore();
  const Card = new S.OfObject.Builder().name("Card").ref().properties(text("name")).create();
  local.register(Card);
  const Note = new S.OfObject.Builder().properties(text("text"), (p) => p.name("card").of(Card)).create();
  assert(equal(Modules.reference(Card), map({ named: { name: "Card" } }))); // named: by name
  const inline = Modules.reference(Note); // unnamed: inline, its members by name where they have one
  assert(equal(inline, map({ object: { properties: [{ name: "text", type: { native: { format: "basic", token: "str" } } },
    { name: "card", type: { named: { name: "Card" } } }] } })));
  assert(Modules.resolve(local, map({ named: { name: "Card" } })) === Card);
  const copy = Modules.resolve(local, inline) as S.OfObject.Data;
  assert(copy !== Note && copy.properties.get("card")?.type === Card && equal(Modules.reference(copy), inline));
  raises(LookupError, () => Modules.resolve(local, map({ named: { name: "Nope" } })), "no schema named 'Nope' in the module or the store");
}

## MOD-06 · Descriptions travel with a module, written only where present

In [ ]:
{
  const Meters = S.OfNative.resolve((n) => n.name("Meters").type(Number).description("A length"));
  const Calls = new S.OfRelation.Builder().name("Calls").description("Who called whom").links("caller", "callee").properties(
    (p) => p.name("at").of((t) => t.as_native(String)).description("When")).create();
  const Line = new S.OfObject.Builder().name("Line").ref().description("A phone line").properties(
    (p) => p.name("length").of(Meters).description("Of cable"),
    (p) => p.name("reach").of(new S.OfUnion.Builder().description("Near or far").branches(
      (b) => b.name("near").of(Meters).description("Within reach"), (b) => b.name("far").of(Meters)).create()),
    (p) => p.name("both").of(new S.OfIntersection.Builder().description("Both at once").parts(
      (b) => b.name("a").of(Phone).description("The first"), (b) => b.name("b").of(Phone)).create()),
    (p) => p.name("tags").of((t) => t.as_indexed((i) => i.of(Meters).description("Any number")))
  ).relations((a) => a.name("calls").of(Calls).me("caller").description("Calls made")).create();
  const module = Modules.module(store, [Phone, Meters, Calls, Line]) as any;
  assert(equal(entries(module).slice(1), map([ // after Phone, as in MOD-02
    { name: "Meters", schema: { native: { format: "basic", token: "float", description: "A length" } } },
    { name: "Calls", schema: { relation: {
      links: ["caller", "callee"], properties: [{ name: "at", type: STR, description: "When" }],
      description: "Who called whom" } } },
    { name: "Line", schema: { object: {
      properties: [
        { name: "length", type: named("Meters"), description: "Of cable" },
        { name: "reach", type: { union: { branches: [
          { name: "near", type: named("Meters"), description: "Within reach" },
          { name: "far", type: named("Meters") }], description: "Near or far" } } },
        { name: "both", type: { intersection: { parts: [
          { name: "a", type: named("Phone"), description: "The first" },
          { name: "b", type: named("Phone") }], description: "Both at once" } } },
        { name: "tags", type: { indexed: { item: named("Meters"), description: "Any number" } } }],
      adjacencies: [{ name: "calls", relation: named("Calls"), me: "caller", description: "Calls made" }],
      ref: true, description: "A phone line" } } }])));
  const back = Modules.schemas(store, JSON.FromJSON(B)(S.Module.Schema, JSON.ToJSON(store)(S.Module.Schema, module))) as Map<string, any>;
  assert(JSON.ToJSON(store)(S.Module.Schema, Modules.module(store, back.values()) as any) === JSON.ToJSON(store)(S.Module.Schema, module));
  const line = back.get("Line");
  assert(back.get("Meters").equals(Meters) && back.get("Calls").description === "Who called whom" && line.description === "A phone line");
  assert(back.get("Calls").properties.get("at")?.description === "When" && line.properties.get("length")?.description === "Of cable");
  assert(line.properties.get("reach")?.type.properties.get("near")?.description === "Within reach" && line.adjacencies.get("calls")?.description === "Calls made");
  assert(line.properties.get("both")?.type.description === "Both at once" && line.properties.get("tags")?.type.description === "Any number");
  assert([...back.values()].every((schema) => schema.validate().length === 0));
}